# Association Analysis

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import glob

from micom import load_pickle
from micom.qiime_formats import load_qiime_model_db, load_qiime_medium
from micom.workflows import grow, save_results, load_results
from micom.viz import *
from micom.measures import *
from micom.workflows.core import GrowthResults
from micom.viz import association

In [2]:
minimal_imports_lf0 = load_results("results/lf0/growth_minimal_imports.zip")
minimal_imports_hf0 = load_results("results/hf0/growth_minimal_imports.zip")

In [3]:
growth_lf0 = minimal_imports_lf0.growth_rates
growth_lf0['diet'] = 'LF/HPP'
growth_hf0 = minimal_imports_hf0.growth_rates
growth_hf0['diet'] = 'HF/HS'
growth = pd.concat([growth_lf0, growth_hf0], axis=0, ignore_index=True)
phenotype = pd.DataFrame({
    'sample_id':growth.sample_id,
    'diet':growth.diet
    })
phenotype.drop_duplicates(subset='sample_id', inplace=True)
phenotype.set_index('sample_id', inplace=True)
growth

,abundance,growth_rate,reactions,metabolites,taxon,tradeoff,sample_id,diet
0,0.097084,1.231692e-14,2381,1461,B_caccae,0.6,mouse_01,LF/HPP
1,0.444060,3.826183e-02,2779,1702,B_cellulosilyticus_WH2,0.6,mouse_01,LF/HPP
2,0.139322,1.136146e-22,2476,1526,B_ovatus,0.6,mouse_01,LF/HPP
3,0.093460,1.196529e-01,2524,1549,B_thetaiotaomicron,0.6,mouse_01,LF/HPP
4,0.035084,1.535031e-22,2418,1500,B_uniformis,0.6,mouse_01,LF/HPP
...,...,...,...,...,...,...,...,...
160,0.004272,1.445010e-06,920,850,C_aerofaciens,0.6,mouse_15,HF/HS
161,0.019892,4.351062e-07,2049,1394,C_scindens,0.6,mouse_15,HF/HS
162,0.000125,2.261928e-04,1980,1313,C_spiroforme,0.6,mouse_15,HF/HS
163,0.034353,9.585099e-03,2519,1570,P_distasonis,0.6,mouse_15,HF/HS


In [4]:
print(phenotype)

             diet
sample_id        
mouse_01   LF/HPP
mouse_02   LF/HPP
mouse_03   LF/HPP
mouse_04   LF/HPP
mouse_05   LF/HPP
mouse_06   LF/HPP
mouse_07   LF/HPP
mouse_08    HF/HS
mouse_09    HF/HS
mouse_10    HF/HS
mouse_11    HF/HS
mouse_12    HF/HS
mouse_13    HF/HS
mouse_14    HF/HS
mouse_15    HF/HS


In [5]:
exchanges_lf0 = minimal_imports_lf0.exchanges
growth_lf0['diet'] = 'LF/HPP'
exchanges_hf0 = minimal_imports_hf0.exchanges
growth_hf0['diet'] = 'HF/HS'
exchanges = pd.concat([exchanges_lf0, exchanges_hf0], axis=0, ignore_index=True)
exchanges

,taxon,sample_id,tolerance,reaction,flux,abundance,metabolite,direction
0,medium,mouse_01,0.000001,EX_h2o_m,0.345596,NaN,h2o_m,export
1,medium,mouse_01,0.000001,EX_fol_m,-0.000055,NaN,fol_m,import
2,medium,mouse_01,0.000001,EX_q8_m,-0.000007,NaN,q8_m,import
3,medium,mouse_01,0.000001,EX_cobalt2_m,-0.000200,NaN,cobalt2_m,import
4,medium,mouse_01,0.000001,EX_zn2_m,-0.000200,NaN,zn2_m,import
...,...,...,...,...,...,...,...,...
8422,C_aerofaciens,mouse_15,0.000001,EX_lac_D(e),599.590042,0.004272,lac_D[e],export
8423,C_aerofaciens,mouse_15,0.000001,EX_hxan(e),-0.409947,0.004272,hxan[e],import
8424,C_aerofaciens,mouse_15,0.000001,EX_fru(e),-100.000000,0.004272,fru[e],import
8425,C_aerofaciens,mouse_15,0.000001,EX_h(e),-100.000000,0.004272,h[e],import


In [6]:
annotations = minimal_imports_lf0.annotations
annot = annotations.copy()
annotations

,metabolite,name,molecular_weight,C_number,N_number,hmdb,inchi,kegg.compound,pubchem.compound,reaction
0,12ppd_S[e],"(S)-propane-1,2-diol",76.09442,3,0,HMDB06213,"InChI=1S/C3H8O2/c1-3(5)2-4/h3-5H,2H2,1H3/t3-/m...",C02917,NaN,EX_12ppd_S(e)
1,2obut[e],2-Oxobutanoate,101.08070,4,0,HMDB00005,"InChI=1S/C4H6O3/c1-2-3(5)4(6)7/h2H2,1H3,(H,6,7...",C00109,58.0,EX_2obut(e)
2,4abut[e],4-Aminobutanoate,103.11976,4,1,HMDB00112,"InChI=1S/C4H9NO2/c5-3-1-2-4(6)7/h1-3,5H2,(H,6,7)",C00334,119.0,EX_4abut(e)
3,4hbz[e],4-hydroxybenzoate,137.11280,7,0,HMDB00500,"InChI=1S/C7H6O3/c8-6-3-1-5(2-4-6)7(9)10/h1-4,8...",C00156,NaN,EX_4hbz(e)
4,MGlcn10[e],mucin-type O-glycan No 10,NaN,44,4,NaN,NaN,NaN,NaN,EX_MGlcn10(e)
...,...,...,...,...,...,...,...,...,...,...
1399,mqn7_m,Menaquinone 7,648.99916,46,0,NaN,NaN,C00828,4086.0,EX_mqn7_m
1400,n2_m,Nitrogen,28.01340,0,2,NaN,NaN,NaN,NaN,EX_n2_m
1401,sbt_D_m,D-glucitol,182.17176,6,0,HMDB00247,InChI=1S/C6H14O6/c7-1-3(9)5(11)6(12)4(10)2-8/h...,C00794,5780.0,EX_sbt_D_m
1402,thm_m,Thiamin,265.35458,12,4,HMDB00235,InChI=1S/C12H17N4OS/c1-8-11(3-4-17)18-7-16(8)6...,C00378,1130.0,EX_thm_m


In [7]:
minimal_imports = GrowthResults(growth, exchanges, annotations)
print(minimal_imports)

GrowthResults(growth_rates=     abundance   growth_rate  reactions  metabolites                   taxon  \
0     0.097084  1.231692e-14       2381         1461                B_caccae   
1     0.444060  3.826183e-02       2779         1702  B_cellulosilyticus_WH2   
2     0.139322  1.136146e-22       2476         1526                B_ovatus   
3     0.093460  1.196529e-01       2524         1549      B_thetaiotaomicron   
4     0.035084  1.535031e-22       2418         1500             B_uniformis   
..         ...           ...        ...          ...                     ...   
160   0.004272  1.445010e-06        920          850           C_aerofaciens   
161   0.019892  4.351062e-07       2049         1394              C_scindens   
162   0.000125  2.261928e-04       1980         1313            C_spiroforme   
163   0.034353  9.585099e-03       2519         1570            P_distasonis   
164   0.009636  5.057161e-06       1225         1088                 R_obeum   

     tradeof

In [8]:
association.plot_association(
    minimal_imports, 
    phenotype.diet, 
    'binary', 
    'diet', 
    'results/associations.html'
    )